# QA - Valida??o de Tiles de Mosaico/C10

**Objetivo:** Validar pareamento e consist?ncia dos tiles de mosaico e m?scara C10.

**Entradas esperadas:**
- /content/drive/MyDrive/GEE_Exports/unet_mt_2023_mosaic_*.tif
- /content/drive/MyDrive/GEE_Exports/unet_mt_2023_c10mask_*.tif

**Sa?das geradas:**
- Diagn?sticos de cobertura, pareamento e estat?sticas b?sicas

**Posi??o no fluxo:** Ponto de verifica??o de QA antes do pr?-processamento de shards.


In [1]:
!pip -q install rasterio numpy pandas tqdm

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, re, glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import rasterio

# ajuste se você exportou para outra pasta
DRIVE_FOLDER = "/content/drive/MyDrive/GEE_Exports"
PREFIX = "unet_mt_2023"

mosaic_glob = os.path.join(DRIVE_FOLDER, f"{PREFIX}_mosaic_x*_y*.tif")
mask_glob   = os.path.join(DRIVE_FOLDER, f"{PREFIX}_c10mask_x*_y*.tif")

mosaic_files = sorted(glob.glob(mosaic_glob))
mask_files   = sorted(glob.glob(mask_glob))

print("Mosaic tiles:", len(mosaic_files))
print("Mask tiles  :", len(mask_files))

assert len(mosaic_files) > 0, "Nenhum mosaic encontrado. Verifique DRIVE_FOLDER/PREFIX."
assert len(mask_files) > 0, "Nenhuma mask encontrada. Verifique DRIVE_FOLDER/PREFIX."

Mosaic tiles: 169
Mask tiles  : 169


In [3]:
# extrai "x-50p00_y-10p00" etc
tile_key_re = re.compile(r"_x([^_]+)_y([^_]+)\.tif$", re.IGNORECASE)

def tile_key(path: str):
  m = tile_key_re.search(os.path.basename(path))
  if not m:
    return None
  return (m.group(1), m.group(2))

mosaic_map = {tile_key(p): p for p in mosaic_files}
mask_map   = {tile_key(p): p for p in mask_files}

all_keys = sorted(set(mosaic_map.keys()) | set(mask_map.keys()))
missing_mosaic = [k for k in all_keys if k not in mosaic_map]
missing_mask   = [k for k in all_keys if k not in mask_map]

print("Tiles sem mosaic:", len(missing_mosaic))
print("Tiles sem mask  :", len(missing_mask))

if missing_mosaic[:5]:
  print("Exemplos sem mosaic:", missing_mosaic[:5])
if missing_mask[:5]:
  print("Exemplos sem mask:", missing_mask[:5])

# opcional: falhar se houver inconsistências
assert len(missing_mosaic) == 0, "Existem tiles sem mosaic."
assert len(missing_mask) == 0, "Existem tiles sem mask."


Tiles sem mosaic: 0
Tiles sem mask  : 0


In [4]:
def read_meta(path):
  with rasterio.open(path) as ds:
    meta = {
      "path": path,
      "crs": str(ds.crs),
      "width": ds.width,
      "height": ds.height,
      "count": ds.count,
      "dtype": ds.dtypes[0],
      "transform": ds.transform,
      "res_x": abs(ds.transform.a),
      "res_y": abs(ds.transform.e),
      "bounds": ds.bounds,
      "nodata": ds.nodata,
    }
  return meta

def sample_stats_mosaic(path, max_samples=200_000, seed=42):
  rng = np.random.default_rng(seed)
  with rasterio.open(path) as ds:
    h, w = ds.height, ds.width
    n = min(max_samples, h*w)
    rows = rng.integers(0, h, size=n)
    cols = rng.integers(0, w, size=n)

    stats = []
    for b in range(1, ds.count + 1):
      vals = ds.read(b)[rows, cols]
      nodata = ds.nodata
      if nodata is not None:
        vals = vals[vals != nodata]
      if vals.size == 0:
        stats.append({"band": b, "min": np.nan, "p1": np.nan, "p50": np.nan, "p99": np.nan, "max": np.nan, "n": 0})
        continue
      stats.append({
        "band": b,
        "min": float(np.min(vals)),
        "p1":  float(np.quantile(vals, 0.01)),
        "p50": float(np.quantile(vals, 0.50)),
        "p99": float(np.quantile(vals, 0.99)),
        "max": float(np.max(vals)),
        "n": int(vals.size),
      })
    return stats

def sample_stats_mask(path, max_samples=200_000, seed=42):
  rng = np.random.default_rng(seed)
  with rasterio.open(path) as ds:
    assert ds.count == 1, "Mask deve ter 1 banda"
    h, w = ds.height, ds.width
    n = min(max_samples, h*w)
    rows = rng.integers(0, h, size=n)
    cols = rng.integers(0, w, size=n)

    vals = ds.read(1)[rows, cols]
    nodata = ds.nodata
    if nodata is not None:
      vals = vals[vals != nodata]

    uniq = np.unique(vals)
    frac_ones = float(np.mean(vals == 1)) if vals.size else np.nan
    frac_zeros = float(np.mean(vals == 0)) if vals.size else np.nan

    return {
      "uniq": uniq.tolist(),
      "frac_ones": frac_ones,
      "frac_zeros": frac_zeros,
      "n": int(vals.size),
    }

In [5]:
import rasterio, rasterio.env
import subprocess, textwrap, os

print("rasterio:", rasterio.__version__)
print("gdal (rasterio):", rasterio.__gdal_version__)

# gdalinfo do sistema (se existir)
!gdalinfo --version || true

p = "/content/drive/MyDrive/GEE_Exports/unet_mt_2023_mosaic_x-50p00_y-11p00.tif"
print("exists:", os.path.exists(p), "sizeMB:", os.path.getsize(p)/1024/1024)

# tenta gdalinfo nesse tile (melhor diagnóstico que rasterio)
!gdalinfo -mm -stats "{p}" | head -n 60

rasterio: 1.5.0
gdal (rasterio): 3.12.1
/bin/bash: line 1: gdalinfo: command not found
exists: True sizeMB: 0.9508790969848633
/bin/bash: line 1: gdalinfo: command not found


In [1]:
import random
import rasterio as rio
from rasterio.windows import Window

random.seed(42)

FOLDER = "/content/drive/MyDrive/GEE_Exports"
mosa = sorted(glob.glob(os.path.join(FOLDER, "unet_mt_2023_mosaic_*.tif")))
mask = sorted(glob.glob(os.path.join(FOLDER, "unet_mt_2023_c10mask_*.tif")))

def key(p):
    return os.path.basename(p).replace(".tif","").split("unet_mt_2023_")[1]

mosa_d = {key(p).replace("mosaic_",""): p for p in mosa}
mask_d = {key(p).replace("c10mask_",""): p for p in mask}

keys = list(set(mosa_d) & set(mask_d))
for k in random.sample(keys, 3):
    mp, kp = mosa_d[k], mask_d[k]
    with rio.open(mp) as ds_m, rio.open(kp) as ds_k:
        print("\n", k)
        print("mosaic:", ds_m.count, ds_m.dtypes[0], ds_m.width, ds_m.height)
        print("mask  :", ds_k.count, ds_k.dtypes[0], ds_k.width, ds_k.height)
        # lê um patch 256x256 no canto superior esquerdo
        w = Window(0, 0, min(256, ds_m.width), min(256, ds_m.height))
        patch = ds_m.read(1, window=w)
        mpatch = ds_k.read(1, window=w)
        print("mosaic min/max:", float(patch.min()), float(patch.max()))
        print("mask unique:", set(mpatch.ravel()[:1000]))


NameError: name 'glob' is not defined

In [ ]:
!apt-get -qq update
!apt-get -qq install -y gdal-bin

In [ ]:
!gdalinfo /content/drive/MyDrive/GEE_Exports/unet_mt_2023_mosaic_x-50p00_y-11p00.tif | head -n 80